<a href="https://colab.research.google.com/github/jmorrone/biomed-multi-view/blob/main/mmelon_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Install: First clone repo and then install, with requirements.txt packages installed as in README

Tested with the following CUDA:

nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2024 NVIDIA Corporation
Built on Thu_Jun__6_02:18:23_PDT_2024
Cuda compilation tools, release 12.5, V12.5.82
Build cuda_12.5.r12.5/compiler.34385749_0

In [ ]:
! git clone https://github.com/jmorrone/biomed-multi-view.git

In [ ]:
! pip install git+https://github.com/jmorrone/biomed-multi-view.git

Note that there are still some errors due to version issues between what is required by biomed-multi-view and the newer libraries pre-installed, but not used (presumably) by biomed-multi-view.  Also, the notebook must be restarted after install.

In [ ]:
import os
os.chdir('biomed-multi-view')

In [ ]:
# These packages are installed above.  Command is commented out
# ! pip install -e .['dev']

Note that fast transformers may take a long time to build (> 15 minutes) on some instances

In [ ]:
!pip install -r requirements.txt

# Test torch, numpy versions, GPU

In [ ]:
import torch
import numpy

In [ ]:
torch.__version__

In [ ]:
numpy.__version__

In [ ]:
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"Number of GPUs: {torch.cuda.device_count()}")
    # To use the GPU for your tensors and model:
    device = torch.device("cuda")
    print(f"Active device: {device}")
else:
    print("Using CPU")

# Test Cells from MMELON API notebook

In [ ]:
from bmfm_sm.api.smmv_api import SmallMoleculeMultiViewModel, PredictionIterator
from bmfm_sm.core.data_modules.namespace import LateFusionStrategy
from bmfm_sm.api.dataset_registry import DatasetRegistry

from dataclasses import asdict
from itertools import islice
import pandas as pd
import os

In [ ]:
pd.DataFrame(DatasetRegistry.get_instance().get_collection('MoleculeNet'))

In [ ]:
model = SmallMoleculeMultiViewModel.from_pretrained(LateFusionStrategy.ATTENTIONAL,
                                                    model_path='ibm/biomed.sm.mv-te-84m',
                                                    huggingface=True)

In [ ]:
example_smiles = "CC(C)CC1=CC=C(C=C1)C(C)C(=O)O"
example_emb = SmallMoleculeMultiViewModel.get_embeddings(
    smiles=example_smiles,
    model_path="ibm/biomed.sm.mv-te-84m",
    huggingface=True,
)
print(example_emb.shape)

In [ ]:
dataset_registry = DatasetRegistry()
ds = dataset_registry.get_dataset_info('BACE')

In [ ]:
finetuned_model_ds = SmallMoleculeMultiViewModel.from_finetuned(
    ds,
    model_path="ibm/biomed.sm.mv-te-84m-MoleculeNet-ligand_scaffold-BACE-101",
    inference_mode=True,
    huggingface=True
)

In [ ]:
# Get predictions
prediction = SmallMoleculeMultiViewModel.get_predictions(
    example_smiles, ds, finetuned_model=finetuned_model_ds
)
print(prediction)

In [ ]:
ds = dataset_registry.get_dataset_info('SOLUBILITY')

finetuned_model_ds = SmallMoleculeMultiViewModel.from_finetuned(
    ds,
    model_path="ibm-research/biomed.sm.mv-te-84m-ComputationalADME-random-SOLUBILITY-101",
    inference_mode=True,
    huggingface=True
)

prediction = SmallMoleculeMultiViewModel.get_predictions(
    example_smiles, ds, finetuned_model=finetuned_model_ds
)
print(prediction)